In [ ]:
import mysql.connector
from datetime import datetime

# 1. Conexión a MySQL
conexion = mysql.connector.connect(
    host="localhost",
    user="root",
    password="TU PASSWORD AQUÍ",
    database="rifa_db"
)

cursor = conexion.cursor()

# 2. Inputs interactivos
numero_boleto = input("Ingrese número de boleto: ")
comprador = input("Ingrese nombre del comprador: ")

# 3. Validar formato del número (rellenar con ceros a la izquierda)
numero_boleto = numero_boleto.zfill(4)   # Ej: "45" → "0045"

# 4. Verificar si el boleto existe y su estado
cursor.execute("SELECT estado FROM boletos WHERE numero = %s", (numero_boleto,))
resultado = cursor.fetchone()

if resultado is None:
    print(f"⚠️ El boleto {numero_boleto} no existe en la base.")
else:
    estado_actual = resultado[0]
    if estado_actual == "Vendido":
        print(f"❌ El boleto {numero_boleto} ya está vendido.")
    else:
        # 5. Actualizar el boleto con fecha de compra actual
        fecha_actual = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        cursor.execute("""
            UPDATE boletos
            SET comprador = %s, estado = 'Vendido', fecha_Compra = %s
            WHERE numero = %s AND estado = 'Disponible'
        """, (comprador, fecha_actual, numero_boleto))
        conexion.commit()

        print(f"✅ Venta cargada con éxito: Boleto {numero_boleto} vendido a {comprador} el {fecha_actual}")

# 6. Mostrar resumen de la tabla
cursor.execute("SELECT COUNT(*) FROM boletos")
total = cursor.fetchone()[0]
print(f"📊 Total de boletos en la tabla: {total}")

cursor.execute("SELECT COUNT(*) FROM boletos WHERE estado = 'Disponible'")
disponibles = cursor.fetchone()[0]
print(f"🎟️ Boletos disponibles: {disponibles}")

cursor.execute("SELECT COUNT(*) FROM boletos WHERE estado = 'Vendido'")
vendidos = cursor.fetchone()[0]
print(f"💰 Boletos vendidos: {vendidos}")

cursor.close()
conexion.close()


✅ Venta cargada con éxito: Boleto 0003 vendido a TERE A el 2026-05-23 17:49:10
📊 Total de boletos en la tabla: 1001
🎟️ Boletos disponibles: 993
💰 Boletos vendidos: 8
